### Import the necessary database

In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
#In[2]:
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprosess

In [2]:
import src.slurm_cluster as scluster
client, scluster = scluster.init_dask_slurm_cluster()

/home/m/m301036/.conda/envs/mykernel/lib/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34941 instead
  warnings.warn(


#!/usr/bin/env bash

#SBATCH -J dask-worker
#SBATCH -e /scratch/m/m301036/dask_logs//dask-worker-%J.err
#SBATCH -o /scratch/m/m301036/dask_logs//dask-worker-%J.out
#SBATCH -p compute
#SBATCH -A mh0033
#SBATCH -n 1
#SBATCH --cpus-per-task=64
#SBATCH --mem=256G
#SBATCH -t 06:00:00

/home/m/m301036/.conda/envs/mykernel/bin/python -m distributed.cli.dask_worker tcp://10.128.10.130:44057 --name dummy-name --nthreads 1 --memory-limit 4.00GiB --nworkers 64 --nanny --death-timeout 60 --local-directory /scratch/m/m301036/dask_temp/ --interface ib0



In [3]:
print(client)

<Client: 'tcp://10.128.10.130:44057' processes=64 threads=64, memory=256.00 GiB>


In [4]:
dir_ICV_STD = "/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/OBS_ICV_std/"
len_segments = np.arange(10, 74, 1)
import os
start_year = 1950
end_year   = 2022
min_length = 10
period_starts = list(range(end_year - min_length + 1, start_year - 1, -1))  # 2013..1950
period_labels = [f"{by}-{end_year}" for by in period_starts]

# Load and tag each single-L file
datasets = []
for L, period_label in zip(len_segments, period_labels):
    ds_L = xr.open_mfdataset(
        os.path.join(dir_ICV_STD, f"OBS_ICV_MK_trend_STD_L{L}.nc"),
        chunks={"lat": 10, "lon": 10},
    ).assign_coords(period=[period_label])  # tag length
    datasets.append(ds_L)

# Concatenate over trend_length and add period labels
ICV_STD_MME_HadCRUT5_all = xr.concat(datasets, dim="period")
ICV_STD_MME_HadCRUT5_all = ICV_STD_MME_HadCRUT5_all.assign_coords(
    period=("period", period_labels)
)

print(ICV_STD_MME_HadCRUT5_all)

<xarray.Dataset>
Dimensions:        (trend_length: 64, lat: 90, lon: 180, period: 64)
Coordinates:
  * trend_length   (trend_length) int64 10 11 12 13 14 15 ... 68 69 70 71 72 73
  * lat            (lat) float64 -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
  * lon            (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
  * period         (period) <U9 '2013-2022' '2012-2022' ... '1950-2022'
Data variables:
    icv_trend_std  (period, trend_length, lat, lon) float64 dask.array<chunksize=(1, 64, 10, 10), meta=np.ndarray>


In [5]:
ICV_STD_MME_HadCRUT5_all

<xarray.Dataset>
Dimensions:        (trend_length: 64, lat: 90, lon: 180, period: 64)
Coordinates:
  * trend_length   (trend_length) int64 10 11 12 13 14 15 ... 68 69 70 71 72 73
  * lat            (lat) float64 -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
  * lon            (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
  * period         (period) <U9 '2013-2022' '2012-2022' ... '1950-2022'
Data variables:
    icv_trend_std  (period, trend_length, lat, lon) float64 dask.array<chunksize=(1, 64, 10, 10), meta=np.ndarray>

In [7]:
# output the netcdf file
dir_out = "/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/OBS_ICV_std/concatenate/"
output_path = dir_out + "OBS_ICV_MK_trend_STD_1950_2022_sliding.nc"
ICV_STD_MME_HadCRUT5_all.to_netcdf(output_path)

### Plot the trend of the ICV patterns

In [8]:
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
# plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True
plt.rcParams['savefig.transparent'] = True

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap

In [9]:
def plot_trend(trend_data, lats, lons, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - trend_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """
# Create a new figure/axis if none is provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(20, 15), subplot_kw={'projection': ccrs.Robinson()})
        ax.set_global()
        
    contour_obj = ax.contourf(lons, lats, trend_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree(central_longitude=0))
    # Plot significance masks with different hatches
    # ax.contourf(lons, lats, significance_mask, levels=[0.05, 1.0],hatches=['///'], colors='none', transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, linewidth=1, color='gray', alpha=0.35)

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 16}
    gl.ylabel_style = {'size': 16}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    ax.set_title(title, loc='center', fontsize=24, pad=5.0)

    return contour_obj

In [10]:
# calculate the ensemble mean of the trend pattern of each interval of segments;
#     and save the ensemble mean of the trend pattern of each interval of segments to the dataset
# for segment_length in segment_lengths:
    # ds_combined[f'ICV_segments_{segment_length}yr_trend_mean'] = ds_combined[f'ICV_segments_{segment_length}yr_trend'].mean(dim='segment')

In [11]:
# define an asymmetric colormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm

intervals = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]

# Normalizing the intervals to [0, 1]
min_interval = min(intervals)
max_interval = max(intervals)
normalized_intervals = [(val - min_interval) / (max_interval - min_interval) for val in intervals]

# colors = ['#2616D3', '#005EFF', '#0084FF', '#00A2FF', '#00BCDB', (1.0, 1.0, 1.0, 1.0),(1.0, 1.0, 1.0, 1.0),(1.0, 0.8, 0.5, 1.0),
#     (1.0, 0.803921568627451, 0.607843137254902, 1.0), (1.0, 0.6000000000000001, 0.20000000000000018, 1.0),(1.0, 0.4039215686274509, 0.0, 1.0),(0.8999999999999999, 0.19999999999999996, 0.0, 1.0),
#     (0.7470588235294118, 0.0, 0.0, 1.0), (0.6000000000000001, 0.0, 0.0, 1.0),(0.44705882352941173, 0.0, 0.0, 1.0),(0.30000000000000004, 0.0, 0.0, 1.0),(0.14705882352941177, 0.0, 0.0, 1.0),
#     (0.0, 0.0, 0.0, 1.0)]

# Creating a list of tuples with normalized positions and corresponding colors
# color_list = list(zip(normalized_intervals, colors))

# # Create the colormap
# custom_cmap = LinearSegmentedColormap.from_list('my_custom_cmap', color_list)

# # Create a normalization
# norm = Normalize(vmin=min_interval, vmax=max_interval)

In [12]:
import seaborn as sns
import palettable
from palettable.colorbrewer.diverging import RdBu_11_r
import matplotlib.colors as mcolors

cmap = mcolors.ListedColormap(palettable.cmocean.sequential.Amp_20.mpl_colors)

In [18]:
print(client)

<Client: 'tcp://10.128.10.130:44665' processes=128 threads=128, memory=512.00 GiB>


In [19]:
# trend_data['15yr']

In [13]:
from matplotlib.backends.backend_pdf import PdfPages

import cartopy.util as cutil

# Extract data from the xarray dataset
lat = ICV_STD_MME_HadCRUT5_all['lat'].values
lon = ICV_STD_MME_HadCRUT5_all['lon'].values
periods = ICV_STD_MME_HadCRUT5_all['period'].values

# Parameters for the PDF pages
num_subplots_x = 5
num_subplots_y = 3
num_plots_per_page = num_subplots_x * num_subplots_y
figsize_x = 25
figsize_y = 15

levels = np.arange(0.0, 1.1, 0.1)

with PdfPages("./ICV_STD_sliding_trends_7LEs.pdf") as pdf:
    for start_page in range(0, len(periods), num_plots_per_page):
        fig, axes = plt.subplots(
            num_subplots_y,
            num_subplots_x,
            figsize=(figsize_x, figsize_y),
            subplot_kw={"projection": ccrs.Robinson(central_longitude=180)},
        )
        fig.subplots_adjust(hspace=0.4, wspace=0.4)

        contour_obj = None

        for i in range(num_plots_per_page):
            idx = start_page + i
            if idx >= len(periods):
                break

            period = periods[idx]
            data = ICV_STD_MME_HadCRUT5_all['icv_trend_std'].sel(period=period).mean(dim='trend_length').compute()

            iy = i // num_subplots_x
            ix = i % num_subplots_x
            ax = axes[iy, ix]

            data_cyc, lon_cyc = cutil.add_cyclic_point(data.values, coord=lon)

            contour_obj = plot_trend(
                data_cyc,
                lat,
                lon_cyc,
                levels=levels,
                extend='max',
                cmap=cmap,
                title="",
                ax=ax,
                show_xticks=False,
                show_yticks=False,
            )
            ax.set_title(f"Trend for {period}", fontsize=18)

        # Colorbar shared per page
        cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.02])
        cbar = plt.colorbar(contour_obj, cax=cbar_ax, orientation="horizontal", extend='max')
        cbar.set_label("ICV STD trend (°C per decade)", fontsize=16)

        pdf.savefig(fig)
        plt.close(fig)

/tmp/ipykernel_3029998/880198402.py:61: MatplotlibDeprecationWarning: The 'extend' parameter to Colorbar has no effect because it is overridden by the mappable; it is deprecated since 3.3 and will be removed two minor releases later.
  cbar = plt.colorbar(contour_obj, cax=cbar_ax, orientation="horizontal", extend='max')
/tmp/ipykernel_3029998/880198402.py:61: MatplotlibDeprecationWarning: The 'extend' parameter to Colorbar has no effect because it is overridden by the mappable; it is deprecated since 3.3 and will be removed two minor releases later.
  cbar = plt.colorbar(contour_obj, cax=cbar_ax, orientation="horizontal", extend='max')
/tmp/ipykernel_3029998/880198402.py:61: MatplotlibDeprecationWarning: The 'extend' parameter to Colorbar has no effect because it is overridden by the mappable; it is deprecated since 3.3 and will be removed two minor releases later.
  cbar = plt.colorbar(contour_obj, cax=cbar_ax, orientation="horizontal", extend='max')
/tmp/ipykernel_3029998/880198402.p

In [14]:
client.close()
scluster.close()